In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# 1. Define your base directory using Pathlib (makes it easy to update later)
BASE_DIR = Path("/Users/wmuheki/Documents/Projects/Analytics/KYC/clean_dumps")

# 2. Load only two DataFrames to save massive amounts of RAM  ---- Change Dataframe Names ‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️
df = pd.read_parquet(BASE_DIR / "df_02_26.parquet")
df_NID = pd.read_parquet(BASE_DIR / "df_NID_02_26.parquet")

# 3. Create lightweight 'views' dynamically based on cleaned 'id_type' column
df_PASS = df[df["id_type"] == "PASSPORT"].reset_index(drop=True)
df_REF = df[df["id_type"] == "REFUGEE_ID"].reset_index(drop=True)
df_COM = df[df["id_type"].isin(["EMPLOYEE_ID", "COMPANY_ID"])].reset_index(drop=True)
df_UN = df[df["id_type"] == "UNKNOWN"].reset_index(drop=True)


In [29]:
import clickhouse_connect

# Connect to ClickHouse with resource limits
client = clickhouse_connect.get_client(
    host='192.168.1.95',
    port=8123,
    username='default',
    password='',
    database='ceir_gold',
    settings={
        'max_execution_time': 120,
        'max_memory_usage': 2000000000,  # 2GB max
        'max_threads': 2,
        'priority': 5
    }
)

# Step 1: Fetch data from the gsma_devices table
gsma_query = """
SELECT
    tac,
    device_name,
    marketing_name,
    brand_name,
    model_name,
    device_type,
    operating_system,
    manufacturer,
    sim_slots,
    year_released
FROM gsma_devices
"""
gsma_df = client.query_df(gsma_query)


# Step 2: Fetch data from the domestic_subscribers table
domestic_query = """
SELECT
    msisdn,
    imsi,
    imei,
    id_number
FROM domestic_subscribers
WHERE last_seen >= now() - INTERVAL 90 DAY
ORDER BY last_seen DESC
LIMIT 10000
"""
domestic_df = client.query_df(domestic_query)

# Step 3: Fetch data from the roamers table
roam_query = """
SELECT
    msisdn,
    imsi,
    imei,
    id_number
FROM roamers
WHERE last_seen >= now() - INTERVAL 90 DAY
ORDER BY last_seen DESC
LIMIT 10000
"""
roam_df = client.query_df(roam_query)


ModuleNotFoundError: No module named 'clickhouse_connect'

In [30]:
# Display the fetched data (optional)
print(f"Domestic Subscribers Data Loaded: {len(domestic_df)} records")
domestic_df.head(10)

NameError: name 'domestic_df' is not defined

In [31]:
print(f"Roamers Data Loaded: {len(roam_df)} records")
roam_df.head(10)

NameError: name 'roam_df' is not defined

In [21]:
print(f"GSMA Devices Data Loaded: {len(gsma_df)} records")
gsma_df.head(1000)

NameError: name 'gsma_df' is not defined

In [22]:

# Check if all MSISDNs start with "256"
if all(domestic_df['msisdn'].str.startswith('256')):
    print("All MSISDNs start with '256'.")

else:
    print("Not all MSISDNs start with '256'. Please check the data.")
     # Replace the first three digits "256" with "0"
    domestic_df['msisdn'] = domestic_df['msisdn'].str.replace('^256', '0', regex=True)
    print("MSISDNs have been updated:")
    print(domestic_df)

NameError: name 'domestic_df' is not defined

In [23]:
kyc_df.head(10)

NameError: name 'kyc_df' is not defined

In [8]:
# Perform Left Join to enrich domestic subscriber data with KYC data.
dom_df = pd.merge(domestic_df, kyc_df, on='msisdn', how='left')

# Display the enriched DataFrame
print("Enriched DataFrame:")
print(dom_df)


NameError: name 'pd' is not defined

In [9]:
dom_df.head(10)

NameError: name 'dom_df' is not defined

In [10]:
domestic_df.head(5)

NameError: name 'domestic_df' is not defined

In [11]:
# Defining prefix rules (prefix, MNO)
PREFIX_RULES = [
    ("020314","TALKIO"), ("020313","TALKIO"), ("020312","TALKIO"),
    ("020311","TALKIO"), ("020310","TALKIO"), ("0728","TALKIO"),
    ("0202494","BCC"), ("0202493","BCC"), ("0202492","BCC"),
    ("0202491","BCC"), ("0202490","BCC"), ("07371","BCC"), ("07370","BCC"),
    ("02054","ROKE"), ("02053","ROKE"), ("02052","ROKE"),
    ("02051","ROKE"), ("02050","ROKE"), ("0734","ROKE"),
    ("02061","HAMILTON"), ("0724","HAMILTON"),
    ("0727","LYCA"), ("0726","LYCA"),
    ("04","UTCL"), ("071","UTCL"),
    ("0207","AIRTEL"), ("0201","AIRTEL"), ("0200","AIRTEL"),
    ("074","AIRTEL"), ("075","AIRTEL"), ("070","AIRTEL"), ("0795","AIRTEL"),
    ("0790","MTN"), ("0791","MTN"), ("0792","MTN"), ("076","MTN"), ("078","MTN"),
    ("077","MTN"), ("03","MTN"),
]

# Split into parallel lists and enforce longest-first to avoid shadowing
prefixes, MNOs = zip(*sorted(PREFIX_RULES, key=lambda x: len(x[0]), reverse=True))
prefixes, MNOs = list(prefixes), list(MNOs)

In [12]:
# Tag each MSISDN with prefix & MNO (vectorized)
msisdn_str = dom_df["msisdn"].astype("string")
mask_list = [msisdn_str.str.startswith(p) for p in prefixes]
dom_df["prefix"] = np.select(mask_list, prefixes, default="UNKNOWN")
dom_df["MNO"]    = np.select(mask_list, MNOs, default="UNKNOWN")

NameError: name 'dom_df' is not defined

In [13]:
dom_df.head(5)

NameError: name 'dom_df' is not defined

In [14]:
# We create District Mapping for further National ID subset Enriching
districts = {
    "001": "APAC", "002": "ARUA", "003": "BUNDIBUGYO", "004": "BUSHENYI", "005": "GULU",
    "006": "HOIMA", "007": "IGANGA", "008": "JINJA", "009": "KABALE", "010": "KABAROLE",
    "011": "KALANGALA", "012": "KAMPALA", "013": "KAMULI", "014": "KAPCHORWA", "015": "KASESE",
    "016": "KIBAALE", "017": "KIBOGA", "018": "KISORO", "019": "KITGUM", "020": "KOTIDO",
    "021": "KUMI", "022": "LIRA", "023": "LUWEERO", "024": "MASAKA", "025": "MASINDI",
    "026": "MBALE", "027": "MBARARA", "028": "MOROTO", "029": "MOYO", "030": "MPIGI",
    "031": "MUBENDE", "032": "MUKONO", "033": "NEBBI", "034": "NTUNGAMO", "035": "PALLISA",
    "036": "RAKAI", "037": "RUKUNGIRI", "038": "SOROTI", "039": "TORORO", "040": "ADJUMANI",
    "041": "BUGIRI", "042": "BUSIA", "043": "KATAKWI", "044": "NAKASONGOLA", "045": "SSEMBABULE",
    "046": "KAMWENGE", "047": "KAYUNGA", "048": "KYENJOJO", "049": "MAYUGE", "050": "PADER",
    "051": "SIRONKO", "052": "WAKISO", "053": "YUMBE", "054": "KABERAMAIDO", "055": "KANUNGU",
    "056": "NAKAPIRIPIRIT", "057": "AMOLATAR", "058": "AMURIA", "059": "BUKWO", "060": "BUTALEJA",
    "061": "IBANDA", "062": "ISINGIRO", "063": "KAABONG", "064": "KALIRO", "065": "KIRUHURA",
    "066": "KOBOKO", "067": "MANAFWA", "068": "MITYANA", "069": "NAKASEKE", "070": "ABIM",
    "071": "AMURU", "072": "BUDAKA", "073": "BULIISA", "074": "DOKOLO", "075": "NAMUTUMBA",
    "076": "OYAM", "077": "MARACHA", "078": "BUDUDA", "079": "BUKEDEA", "080": "LYANTONDE",
    "081": "AMUDAT", "082": "BUIKWE", "083": "BUYENDE", "084": "KYEGEGWA", "085": "LAMWO",
    "086": "OTUKE", "087": "ZOMBO", "088": "ALEBTONG", "089": "BULAMBULI", "090": "BUVUMA",
    "091": "GOMBA", "092": "KIRYANDONGO", "093": "KYANKWANZI", "094": "LUUKA", "095": "NAMAYINGO",
    "096": "NTOROKO", "097": "SERERE", "098": "BUKOMANSIMBI", "099": "BUTAMBALA", "100": "KALUNGU",
    "101": "SHEEMA", "102": "KIBUKU", "103": "KOLE", "104": "KWEEN", "105": "LWENGO",
    "106": "MITOOMA", "107": "NAPAK", "108": "NGORA", "109": "BUHWEJU", "110": "NWOYA",
    "111": "AGAGO", "112": "RUBIRIZI", "113": "KAGADI", "114": "KAKUMIRO", "115": "OMORO",
    "116": "RUBANDA", "117": "BUNYANGABU", "118": "BUTEBO", "119": "KYOTERA", "120": "NAMISINDWA",
    "121": "PAKWACH", "122": "RUKIGA", "123": "BUGWERI", "124": "KAPELEBYONG", "125": "KASSANDA",
    "126": "KIKUUBE", "127": "KWANIA", "128": "NABILATUK", "129": "KALAKI", "130": "KARENGA",
    "131": "KAZO", "132": "KITAGWENDA", "133": "MADI-OKOLLO", "134": "OBONGI", "135": "RWAMPARA",
    "136": "ARUA CITY", "137": "GULU CITY", "138": "JINJA CITY", "139": "FORT PORTAL CITY",
    "140": "MBARARA CITY", "141": "MASAKA CITY", "142": "MBALE CITY", "143": "TEREGO",
    "144": "LIRA CITY", "145": "HOIMA CITY", "146": "SOROTI CITY"
}

In [15]:
# Enriching Data Frame with Age and Gender
current_year = datetime.datetime.now().year

# Gender
dom_df["gender"] = dom_df["id_number"].str[1].map({
    "M": "MALE",
    "F": "FEMALE",
    "X": "UNDEFINED"
})

# Function for birth year
def get_birth_year(nin):
    try:
        yy = int(nin[2:4])
        if yy <= int(str(current_year)[2:]):
            return 2000 + yy
        else:
            return 1900 + yy
    except:
        return None

# Birth Year (as Int64 so it can handle missing values cleanly)
dom_df["birth_year"] = dom_df["id_number"].apply(get_birth_year).astype("Int64")

# age (as Int64)
dom_df["age"] = (current_year - dom_df["birth_year"]).astype("Int64")

# Extract district code and map
dom_df["district"] = (
    dom_df["id_number"].str[4:7].map(districts).fillna("UNKNOWN")
)

NameError: name 'datetime' is not defined

In [16]:
dom_df.head(10)

NameError: name 'dom_df' is not defined